In [ ]:
!nvidia-smi


Sat Jun 27 15:12:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install ultralytics easyocr roboflow --quiet

from ultralytics import YOLO
import torch

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
print(f"GPU      : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.0/250.0 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 151.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.6/299.6 kB 30.0 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/s

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/helmet_detection'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Models will be saved to: {SAVE_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Models will be saved to: /content/drive/MyDrive/helmet_detection


In [ ]:


!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="fV9IQ9Ist8afgpRFhe0b")
project = rf.workspace("onlytusik").project("helmet-detection-fk6is")
version = project.version(1)
dataset = version.download("yolov8")


HELMET_YAML = dataset.location + "/data.yaml"
print(f"Helmet dataset ready → {HELMET_YAML}")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Helmet-detection-1 in yolov8:: 100%|██████████| 16060/16060 [00:03<00:00, 4604.48it/s]


Helmet dataset ready → /content/Helmet-detection-1/data.yaml


In [ ]:
import yaml

with open(HELMET_YAML) as f:
    cfg = yaml.safe_load(f)

print("Classes :", cfg.get('names'))
print("Count   :", cfg.get('nc'))
print("Train   :", cfg.get('train'))
print("Val     :", cfg.get('val'))

Classes : ['with helmet', 'without helmet']
Count   : 2
Train   : ../train/images
Val     : ../valid/images


In [ ]:

from ultralytics import YOLO

helmet_model = YOLO('yolov8n.pt')

helmet_results = helmet_model.train(
    data     = HELMET_YAML,
    epochs   = 50,
    imgsz    = 640,
    batch    = 16,
    name     = 'helmet_detector',
    device   = 0,
    patience = 10,
    save     = True,
    plots    = True,
)

HELMET_BEST = f"{helmet_results.save_dir}/weights/best.pt"
print(f"\n Helmet training done!")
print(f"   Best weights: {HELMET_BEST}")

Ultralytics 8.4.82 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Helmet-detection-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=helmet_detector, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask

In [ ]:
import shutil

helmet_saved = f"{HELMET_BEST}"
drive_saved  = f"{SAVE_DIR}/helmet_model.pt"

shutil.copy(helmet_saved, drive_saved)
print(f" Helmet model saved to Google Drive!")
print(f"   {drive_saved}")

✅ Helmet model saved to Google Drive!
   /content/drive/MyDrive/helmet_detection/helmet_model.pt


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os
print(os.listdir('/content/drive/MyDrive'))


['Classroom', 'PDF_Maker_1741893220236.docx', 'Bidipta Mallik(24051620).pdf', 'Colab Notebooks', 'helmet_detection']


In [ ]:
import os
print(os.listdir('/content/drive/MyDrive/helmet_detection'))

['helmet_model.pt']


In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="fV9IQ9Ist8afgpRFhe0b")
project = rf.workspace("haeun-kim-ri91b").project("license-plate-detection-wienp")
version = project.version(2)
dataset = version.download("yolov8")

PLATE_YAML = dataset.location + "/data.yaml"
print(f"Plate dataset ready → {PLATE_YAML}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.0/250.0 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 92.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 117.2 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.18
    Uninstalling idna-3.18:
      Successfully uninstalled idna-3.18
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to license-plate-detection-2 in yolov8:: 100%|██████████| 11144/11144 [00:01<00:00, 5794.60it/s]

Plate dataset ready → /content/license-plate-detection-2/data.yaml


In [ ]:
!pip install ultralytics easyocr roboflow --quiet

from ultralytics import YOLO
import torch

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
print(f"GPU      : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 114.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.2/296.2 kB 30.3 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
PyTorch  : 2.11.0+cu128
CUDA     : True
GPU      : Tesla T4


In [ ]:
from ultralytics import YOLO

plate_model = YOLO('yolov8n.pt')

plate_results = plate_model.train(
    data     = PLATE_YAML,
    epochs   = 50,
    imgsz    = 640,
    batch    = 16,
    name     = 'plate_detector',
    device   = 0,
    patience = 10,
    save     = True,
    plots    = True,
)

PLATE_BEST = f"{plate_results.save_dir}/weights/best.pt"
print(f"\n Plate training done!")
print(f"   Best weights: {PLATE_BEST}")

Ultralytics 8.4.83 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/license-plate-detection-2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=plate_detector, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overla

In [ ]:
import shutil
plate_saved = f"{SAVE_DIR}/plate_model.pt"
shutil.copy(PLATE_BEST, plate_saved)
print(f" Plate model saved to Google Drive: {plate_saved}")

✅ Plate model saved to Google Drive: /content/drive/MyDrive/helmet_detection/plate_model.pt


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [ ]:
import os
print(os.listdir('/content/drive/MyDrive'))


['Classroom', 'PDF_Maker_1741893220236.docx', 'Bidipta Mallik(24051620).pdf', 'Colab Notebooks', 'helmet_detection']
